# Error Analysis

This notebook performs CPU-only error analysis from saved `results.pkl` files and lightweight protocol metadata. It is intentionally separate from the evaluation notebooks so it can be rerun quickly after new model results are added.

## Output contract

Artifacts are written under `/kaggle/working/error_analysis/` on Kaggle, or `notebook_exports/error_analysis/` locally. If a `results.pkl` lacks evaluated utterance IDs, the notebook reconstructs the eval index without inference and writes patched pickles under `/kaggle/working/results_with_utt_ids/<dataset>/results.pkl`. ASVspoof 2021 additionally checks audio loadability to mirror the eval loader's skip-on-decode-error behavior. ASVspoof 5 now prioritizes the official eval Track 1 protocol and expects eval results to include dataset-level `__metadata__.utt_ids`; older pickles can still fall back to HF tar member order reconstruction. Each dataset also exports `error_analysis.pkl`, CSV tables, plots, and the final cell optionally creates `error_analysis_artifacts.zip` containing both `error_analysis/` and `results_with_utt_ids/` for upload back into the `sdd-survey` Kaggle dataset.

In [ ]:
import math
import os
import pickle
import shutil
import tarfile
import zipfile
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib is not installed; PNG plots will be skipped")

try:
    import soundfile as sf
except ImportError:
    sf = None
    print("soundfile is not installed; ASVspoof 2021 loadability checks will use torchaudio if available")

try:
    import torchaudio
except ImportError:
    torchaudio = None
    print("torchaudio is not installed; ASVspoof 2021 loadability checks will use soundfile only")


BASE_INPUTS = [
    Path("/kaggle/input/datasets/minhbhm/sdd-survey"),
    Path("/kaggle/input/sdd-survey"),
    Path("results"),
]
OUTPUT_ROOT = Path("/kaggle/working/error_analysis") if Path("/kaggle/working").exists() else Path("notebook_exports/error_analysis")
PATCHED_RESULTS_ROOT = Path("/kaggle/working/results_with_utt_ids") if Path("/kaggle/working").exists() else Path("notebook_exports/results_with_utt_ids")
TOP_K_HARD_ERRORS = 25
CREATE_ZIP = True
BACKFILL_UTT_IDS = True
# ASVspoof 5 results in this project were produced from the Hugging Face tar
# stream. If the dev protocol order does not match the saved scores, this flag
# lets the notebook download the required dev tar shards and recover their exact
# member order. This is index reconstruction only: no model loading and no audio
# decoding are performed.
ALLOW_HF_DOWNLOAD = True
RUN_SPECTROGRAMS = False
RANDOM_SEED = 12345
np.random.seed(RANDOM_SEED)
DATASET_STATUS = {}
RESULT_METADATA_KEY = "__metadata__"


DATASETS = {
    "asvspoof19": {
        "display": "ASVspoof 2019 LA",
        "baseline": True,
        "metadata_fields": ["attack", "speaker"],
        "result_paths": [
            PATCHED_RESULTS_ROOT / "asvspoof19" / "results.pkl",
            Path("/kaggle/input/datasets/minhbhm/sdd-survey/asvspoof19/results.pkl"),
            Path("/kaggle/input/sdd-survey/asvspoof19/results.pkl"),
            Path("results/asvspoof19/results.pkl"),
        ],
        "metadata_paths": [
            Path("results/asvspoof19/ASVspoof2019.LA.cm.eval.trl.txt"),
            Path("/kaggle/input/datasets/minhbhm/sdd-survey/asvspoof19/ASVspoof2019.LA.cm.eval.trl.txt"),
            Path("/kaggle/input/sdd-survey/asvspoof19/ASVspoof2019.LA.cm.eval.trl.txt"),
            Path("/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt"),
            Path("/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt"),
        ],
    },
    "asvspoof21": {
        "display": "ASVspoof 2021 DF",
        "metadata_fields": ["codec", "vocoder", "attack", "speaker"],
        "result_paths": [
            PATCHED_RESULTS_ROOT / "asvspoof21" / "results.pkl",
            Path("/kaggle/input/datasets/minhbhm/sdd-survey/asvspoof21/results.pkl"),
            Path("/kaggle/input/sdd-survey/asvspoof21/results.pkl"),
            Path("results/asvspoof21/results.pkl"),
        ],
        "metadata_paths": [
            Path("results/asvspoof21/trial_metadata.txt"),
            Path("/kaggle/input/datasets/minhbhm/sdd-survey/asvspoof21/trial_metadata.txt"),
            Path("/kaggle/input/sdd-survey/asvspoof21/trial_metadata.txt"),
            Path("/kaggle/input/datasets/mohammedabdeldayem/avsspoof-2021/DF-keys-full/keys/DF/CM/trial_metadata.txt"),
            Path("/kaggle/input/avsspoof-2021/DF-keys-full/keys/DF/CM/trial_metadata.txt"),
        ],
        "audio_roots": [
            Path("/kaggle/input/datasets/mohammedabdeldayem/avsspoof-2021"),
            Path("/kaggle/input/avsspoof-2021"),
            Path("/kaggle/input/asvspoof-2021"),
        ],
    },
    "asvspoof5": {
        "display": "ASVspoof 5 Track 1",
        "asv5_split": "eval",
        "metadata_fields": ["attack", "attack_tag", "codec", "condition", "speaker", "gender"],
        "result_paths": [
            PATCHED_RESULTS_ROOT / "asvspoof5" / "results.pkl",
            Path("/kaggle/input/datasets/minhbhm/sdd-survey/asvspoof5/results.pkl"),
            Path("/kaggle/input/sdd-survey/asvspoof5/results.pkl"),
            Path("results/asvspoof5/results.pkl"),
        ],
        "metadata_paths": [
            Path("results/asvspoof5/ASVspoof5.eval.track_1.tsv"),
            Path("/kaggle/input/datasets/minhbhm/sdd-survey/asvspoof5/ASVspoof5.eval.track_1.tsv"),
            Path("/kaggle/input/sdd-survey/asvspoof5/ASVspoof5.eval.track_1.tsv"),
            Path("results/asvspoof5/ASVspoof5.dev.track_1.tsv"),
            Path("/kaggle/input/datasets/minhbhm/sdd-survey/asvspoof5/ASVspoof5.dev.track_1.tsv"),
            Path("/kaggle/input/sdd-survey/asvspoof5/ASVspoof5.dev.track_1.tsv"),
        ],
        "audio_roots": [
            Path("/kaggle/input/datasets/minhbhm/sdd-survey/asvspoof5"),
            Path("/kaggle/input/sdd-survey/asvspoof5"),
            Path("/kaggle/input/asvspoof5"),
            Path("/kaggle/input/asvspoof-5"),
        ],
    },
    "in_the_wild": {
        "display": "In-the-Wild",
        "metadata_fields": ["speaker", "duration_bucket"],
        "result_paths": [
            PATCHED_RESULTS_ROOT / "in_the_wild" / "results.pkl",
            Path("/kaggle/input/datasets/minhbhm/sdd-survey/in_the_wild/results.pkl"),
            Path("/kaggle/input/sdd-survey/in_the_wild/results.pkl"),
            Path("results/in_the_wild/results.pkl"),
        ],
        "metadata_paths": [
            Path("results/in_the_wild/meta.csv"),
            Path("/kaggle/input/datasets/minhbhm/sdd-survey/in_the_wild/meta.csv"),
            Path("/kaggle/input/sdd-survey/in_the_wild/meta.csv"),
            Path("/kaggle/input/datasets/abdallamohamed312/in-the-wild-audio-deepfake/meta.csv"),
            Path("/kaggle/input/in-the-wild-audio-deepfake/meta.csv"),
        ],
    },
}

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PATCHED_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print(f"output root: {OUTPUT_ROOT}")
print(f"patched results root: {PATCHED_RESULTS_ROOT}")

In [ ]:
class NumpyCompatUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module.startswith("numpy._core"):
            module = module.replace("numpy._core", "numpy.core", 1)
        return super().find_class(module, name)


def first_existing(paths):
    for path in paths:
        if Path(path).exists():
            return Path(path)
    return None


def load_pickle_compat(path: Path):
    with open(path, "rb") as handle:
        return NumpyCompatUnpickler(handle).load()


def is_model_result(name, result) -> bool:
    """Return True for model entries and False for dataset metadata entries."""
    if str(name).startswith("__"):
        return False
    return isinstance(result, dict) and "scores" in result and "labels" in result


def save_pickle_compat(obj, path: Path) -> None:
    """Write a pickle using the same protocol used by the evaluation notebooks."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as handle:
        pickle.dump(obj, handle, protocol=pickle.HIGHEST_PROTOCOL)


def get_result_utt_ids(results):
    """Read dataset-level utterance IDs from a patched results.pkl, if present."""
    metadata = results.get(RESULT_METADATA_KEY)
    if isinstance(metadata, dict) and "utt_ids" in metadata:
        return np.asarray(metadata["utt_ids"], dtype=str)
    return None


def result_metadata_matches_config(dataset_key, results, config) -> bool:
    """Check whether dataset metadata is compatible with this analysis config.

    ASVspoof 5 has both older dev-split artifacts and the new official eval-split
    artifacts. When metadata exists for ASVspoof 5, it must explicitly declare
    the configured split/track; otherwise the loader skips that path and tries
    the next candidate instead of silently mixing dev IDs with eval metadata.
    """
    metadata = results.get(RESULT_METADATA_KEY)
    if dataset_key != "asvspoof5" or not isinstance(metadata, dict):
        return True
    return metadata.get("split") == config.get("asv5_split") and metadata.get("track", "track_1") == "track_1"


def labels_from_results(results):
    """Return the first model label vector; metadata entries are ignored."""
    for name, result in results.items():
        if is_model_result(name, result):
            return np.asarray(result["labels"], dtype=np.int64)
    raise ValueError("No model labels found in results.pkl")


def model_names_from_results(results, n_expected):
    """Return model keys with score/label arrays matching the dataset length."""
    names = []
    for model, result in results.items():
        if not is_model_result(model, result):
            continue
        if len(result["scores"]) != n_expected or len(result["labels"]) != n_expected:
            print(f"warning: skipping {model}; length mismatch")
            continue
        names.append(model)
    return names


def compute_eer_threshold(scores, labels):
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int64)
    fpr, tpr, thresholds = roc_curve(labels, scores, pos_label=1)
    fnr = 1.0 - tpr
    idx = int(np.nanargmin(np.abs(fpr - fnr)))
    threshold = float(thresholds[idx])
    if not np.isfinite(threshold):
        finite = thresholds[np.isfinite(thresholds)]
        threshold = float(finite[0]) if len(finite) else 0.5
    return float((fpr[idx] + fnr[idx]) * 50.0), threshold


def compute_operating_metrics(scores, labels, threshold):
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int64)
    pred = (scores >= threshold).astype(np.int64)
    spoof = labels == 0
    bona = labels == 1
    fp = int(np.sum((pred == 1) & spoof))
    fn = int(np.sum((pred == 0) & bona))
    tp = int(np.sum((pred == 1) & bona))
    tn = int(np.sum((pred == 0) & spoof))
    far = fp / max(1, int(np.sum(spoof)))
    frr = fn / max(1, int(np.sum(bona)))
    accuracy = (tp + tn) / max(1, len(labels))
    return {
        "threshold": float(threshold),
        "far": float(far),
        "frr": float(frr),
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "tn": tn,
        "accuracy": float(accuracy),
        "n": int(len(labels)),
    }


def expected_calibration_error(scores, labels, n_bins=10):
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.float64)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    rows = []
    for lo, hi in zip(bins[:-1], bins[1:]):
        if hi == 1.0:
            mask = (scores >= lo) & (scores <= hi)
        else:
            mask = (scores >= lo) & (scores < hi)
        n = int(mask.sum())
        if n == 0:
            rows.append({"bin_left": lo, "bin_right": hi, "n": 0, "mean_score": np.nan, "frac_bonafide": np.nan})
            continue
        mean_score = float(scores[mask].mean())
        frac_bona = float(labels[mask].mean())
        ece += (n / len(scores)) * abs(mean_score - frac_bona)
        rows.append({"bin_left": lo, "bin_right": hi, "n": n, "mean_score": mean_score, "frac_bonafide": frac_bona})
    return float(ece), pd.DataFrame(rows)


def parse_asvspoof19_metadata(path: Path) -> pd.DataFrame:
    rows = []
    with open(path, "r", encoding="utf-8") as handle:
        for line in handle:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            rows.append({
                "speaker": parts[0],
                "utt_id": parts[1],
                "attack": "bonafide" if parts[3] == "-" else parts[3],
                "metadata_label": 1 if parts[4] == "bonafide" else 0,
            })
    return pd.DataFrame(rows)


def parse_asvspoof21_metadata(path: Path) -> pd.DataFrame:
    rows = []
    with open(path, "r", encoding="utf-8") as handle:
        for line in handle:
            parts = line.strip().split()
            if len(parts) < 6:
                continue
            label_text = parts[5]
            if label_text not in ("bonafide", "spoof"):
                continue
            rows.append({
                "speaker": parts[0],
                "utt_id": parts[1],
                "codec": parts[2],
                "source": parts[3] if len(parts) > 3 else "unknown",
                "attack": parts[4] if len(parts) > 4 else "unknown",
                "metadata_label": 1 if label_text == "bonafide" else 0,
                "trim": parts[6] if len(parts) > 6 else "unknown",
                "vocoder": parts[8] if len(parts) > 8 else "unknown",
            })
    return pd.DataFrame(rows)


def parse_asvspoof5_metadata(path: Path) -> pd.DataFrame:
    rows = []
    with open(path, "r", encoding="utf-8") as handle:
        for line in handle:
            parts = line.strip().split()
            if not parts or len(parts) < 5:
                continue
            if len(parts) >= 10:
                speaker, utt_id, gender, codec, codec_q, codec_seed, attack_tag, attack_label, key, tmp = parts[:10]
            else:
                speaker, utt_id, gender, attack_label, key = parts[:5]
                codec, codec_q, codec_seed, attack_tag, tmp = "-", "-", "-", "-", "-"
            if key not in ("bonafide", "spoof"):
                continue
            attack = "bonafide" if attack_label == "bonafide" else attack_label
            codec_norm = "nocodec" if codec == "-" else codec
            rows.append({
                "speaker": speaker,
                "utt_id": utt_id,
                "gender": gender,
                "codec": codec_norm,
                "codec_q": codec_q,
                "codec_seed": codec_seed,
                "attack_tag": "bonafide" if attack_tag == "-" else attack_tag,
                "attack": attack,
                "condition": "bonafide" if key == "bonafide" else ("adversarial" if "adv" in attack.lower() or "adversarial" in attack.lower() else "spoof"),
                "metadata_label": 1 if key == "bonafide" else 0,
            })
    return pd.DataFrame(rows)


def parse_in_the_wild_metadata(path: Path) -> pd.DataFrame:
    raw = pd.read_csv(path)
    lower = {str(c).lower(): c for c in raw.columns}
    file_col = next((lower[k] for k in ("file", "filename", "path", "audio") if k in lower), raw.columns[0])
    label_col = next((lower[k] for k in ("label", "class", "key") if k in lower), raw.columns[-1])
    speaker_col = next((lower[k] for k in ("speaker", "person", "celebrity") if k in lower), None)
    duration_col = next((lower[k] for k in ("duration", "dur", "length") if k in lower), None)

    rows = []
    for _, row in raw.iterrows():
        label_text = str(row[label_col]).strip().lower()
        is_bonafide = label_text in ("bona-fide", "bonafide", "bona_fide", "real", "genuine", "1")
        item = {
            "utt_id": Path(str(row[file_col])).stem,
            "speaker": str(row[speaker_col]) if speaker_col else "unknown",
            "metadata_label": 1 if is_bonafide else 0,
            "label_text": label_text,
        }
        if duration_col is not None:
            item["duration"] = pd.to_numeric(row[duration_col], errors="coerce")
        rows.append(item)
    df = pd.DataFrame(rows)
    if "duration" in df.columns:
        df["duration_bucket"] = pd.cut(
            df["duration"],
            bins=[-np.inf, 2, 5, 10, 20, np.inf],
            labels=["<=2s", "2-5s", "5-10s", "10-20s", ">20s"],
        ).astype(str)
    return df


def build_asvspoof2021_audio_index(config) -> dict[str, Path]:
    """Map ASVspoof 2021 DF eval utterance IDs to FLAC paths.

    The original evaluation dataframe was built by filtering trial_metadata.txt
    to utterances whose FLAC file existed in the three DF eval part folders.
    Match that locator exactly: for each part, walk until the first directory
    containing FLAC files, then use only the immediate filenames in that folder.
    A broad recursive scan can pick up extra files that were never evaluated.
    """
    audio_index = {}
    for root in config.get("audio_roots", []):
        if not root.exists():
            continue
        for part in ("ASVspoof2021_DF_eval_part00", "ASVspoof2021_DF_eval_part01", "ASVspoof2021_DF_eval_part02"):
            part_root = root / part
            if not part_root.exists():
                continue
            audio_dir = None
            for dirpath, _, files in os.walk(part_root):
                if any(name.endswith(".flac") for name in files):
                    audio_dir = Path(dirpath)
                    break
            if audio_dir is None:
                continue
            for name in os.listdir(audio_dir):
                if name.endswith(".flac"):
                    audio_index[name[:-5]] = audio_dir / name
    return audio_index


def list_asvspoof2021_audio_ids(config) -> set[str]:
    """Return ASVspoof 2021 DF eval IDs found by the exact eval locator."""
    return set(build_asvspoof2021_audio_index(config))


def can_load_audio_header(audio_path: Path) -> bool:
    """Return whether an audio file can be decoded by the notebook stack.

    Evaluation datasets mark failed audio loads with `is_error=1` and skip those
    rows when appending scores/labels/utt_ids. This helper mirrors that behavior
    for backfill without running any model. `soundfile.info` is tried first
    because it is lightweight for FLAC; `torchaudio.load` is the fallback and
    matches the eval notebook's actual loader more closely.
    """
    try:
        if sf is not None:
            sf.info(str(audio_path))
            return True
        if torchaudio is not None:
            torchaudio.load(str(audio_path))
            return True
    except Exception:
        return False
    return False


def filter_asvspoof2021_loadable_audio(meta_df: pd.DataFrame, config) -> pd.DataFrame | None:
    """Filter ASVspoof 2021 metadata to files that exist and load successfully.

    The existing-FLAC filter can include a tiny number of corrupt/unreadable
    files. During evaluation those rows were skipped by `WaveformDataset`, so
    this loadability pass is the final CPU-only step needed to recover the exact
    result order. No model inference is performed.
    """
    audio_index = build_asvspoof2021_audio_index(config)
    if not audio_index:
        return None
    rows = []
    failed = []
    for row in meta_df.itertuples(index=False):
        utt_id = str(getattr(row, "utt_id"))
        audio_path = audio_index.get(utt_id)
        if audio_path is None:
            continue
        if can_load_audio_header(audio_path):
            rows.append(row._asdict())
        else:
            failed.append(utt_id)
    if failed:
        print(f"ASVspoof 2021 loadability backfill skipped {len(failed)} unreadable files: {failed[:10]}")
    return pd.DataFrame(rows)


def list_asvspoof5_audio_ids_from_files(config) -> set[str]:
    """List ASVspoof 5 FLAC IDs from attached extracted audio folders.

    This is a lightweight directory scan only. It is used when the protocol file
    contains more rows than the evaluated audio subset.
    """
    audio_ids = set()
    for root in config.get("audio_roots", []):
        if not root.exists():
            continue
        for path in root.rglob("*.flac"):
            audio_ids.add(path.stem)
    return audio_ids


def asvspoof5_tar_paths(config):
    """Find ASVspoof 5 FLAC tar shards in local/Kaggle inputs or HF cache."""
    split = config.get("asv5_split", "dev")
    prefix = {"train": "T", "dev": "D", "eval": "E"}[split]
    roots = list(config.get("audio_roots", [])) + [
        Path("/kaggle/working/asvspoof5_hf_cache"),
        Path("/root/.cache/huggingface"),
    ]
    paths = []
    for root in roots:
        if not root.exists():
            continue
        for pattern in (f"flac_{prefix}_*.tar", f"*flac_{prefix}*.tar"):
            paths.extend(sorted(root.rglob(pattern)))
    # Keep stable order and avoid duplicate paths found through overlapping roots.
    out = []
    seen = set()
    for path in paths:
        key = str(path.resolve())
        if key not in seen:
            out.append(path)
            seen.add(key)
    return out


def download_asvspoof5_hf_tar_paths(config):
    """Download ASVspoof 5 tar shards needed only for index reconstruction.

    The evaluation notebook streamed these tar shards from `jungjee/asvspoof5`.
    To recover the exact evaluated utterance order, we need tar member names in
    shard order. We do not extract or decode audio; `tarfile` only reads member
    headers. The download is optional because the dev shards can be several GB.
    """
    if not ALLOW_HF_DOWNLOAD:
        return [], "ALLOW_HF_DOWNLOAD=False"
    try:
        from huggingface_hub import hf_hub_download, list_repo_files
    except ImportError:
        return [], "huggingface_hub is not installed"

    split = config.get("asv5_split", "dev")
    prefix = {"train": "T", "dev": "D", "eval": "E"}[split]
    repo_id = "jungjee/asvspoof5"
    cache_dir = Path("/kaggle/working/asvspoof5_hf_cache") if Path("/kaggle/working").exists() else Path("notebook_exports/asvspoof5_hf_cache")
    cache_dir.mkdir(parents=True, exist_ok=True)
    shard_names = sorted(
        name for name in list_repo_files(repo_id, repo_type="dataset")
        if name.startswith(f"flac_{prefix}_") and name.endswith(".tar")
    )
    if not shard_names:
        return [], f"no flac_{prefix}_*.tar shards found in {repo_id}"

    local_paths = []
    for shard_name in shard_names:
        local_path = hf_hub_download(
            repo_id=repo_id,
            repo_type="dataset",
            filename=shard_name,
            cache_dir=str(cache_dir),
        )
        local_paths.append(Path(local_path))
    return local_paths, f"downloaded/found {len(local_paths)} flac_{prefix}_*.tar shards from {repo_id}"


def build_asvspoof5_tar_order_index(meta_df: pd.DataFrame, config) -> pd.DataFrame | None:
    """Reconstruct ASVspoof 5 HF-tar evaluation order from tar member order.

    The HF-tar evaluator streams tar shards and scores utterances in member
    order. When results.pkl was produced that way, protocol order is not enough;
    this function lists tar members and maps them back to protocol metadata.
    """
    tar_paths = asvspoof5_tar_paths(config)
    status = f"found {len(tar_paths)} local tar shards"
    if not tar_paths:
        tar_paths, status = download_asvspoof5_hf_tar_paths(config)
    print(f"ASVspoof 5 tar-order backfill: {status}")
    if not tar_paths:
        return None
    meta_by_id = meta_df.drop_duplicates("utt_id").set_index("utt_id", drop=False)
    rows = []
    seen = set()
    for tar_path in tar_paths:
        with tarfile.open(tar_path, "r:*") as tar:
            for member in tar:
                if not member.isfile():
                    continue
                utt_id = Path(member.name).name
                if utt_id.endswith(".flac"):
                    utt_id = utt_id[:-5]
                if utt_id in seen or utt_id not in meta_by_id.index:
                    continue
                rows.append(meta_by_id.loc[utt_id].to_dict())
                seen.add(utt_id)
    if not rows:
        return None
    return pd.DataFrame(rows).reset_index(drop=True)


def validate_eval_index_candidate(name, candidate_df, labels):
    """Validate that a reconstructed index exactly matches saved labels."""
    if candidate_df is None or candidate_df.empty:
        return None, f"{name}: empty candidate"
    if len(candidate_df) != len(labels):
        return None, f"{name}: length {len(candidate_df):,} != result length {len(labels):,}"
    if "metadata_label" in candidate_df.columns:
        candidate_labels = candidate_df["metadata_label"].to_numpy(dtype=np.int64)
        mismatch = int(np.sum(candidate_labels != labels))
        if mismatch:
            return None, f"{name}: label mismatch in {mismatch:,}/{len(labels):,} rows"
    return candidate_df.reset_index(drop=True), f"{name}: length and labels match"


def build_eval_index_from_metadata(dataset_key, config, labels):
    """Rebuild the evaluated utterance index used by results.pkl.

    This function intentionally performs no inference and no audio decoding.
    It only parses protocol rows and, where needed, lists audio filenames to
    recover the exact subset/order that the eval notebooks scored.
    """
    meta_path = first_existing(config.get("metadata_paths", []))
    if meta_path is None:
        raise FileNotFoundError(f"{dataset_key}: protocol metadata file was not found")
    meta_df = METADATA_PARSERS[dataset_key](meta_path)
    labels = np.asarray(labels, dtype=np.int64)
    attempts = []

    direct, status = validate_eval_index_candidate("metadata_order", meta_df, labels)
    attempts.append(status)
    if direct is not None:
        return direct, meta_path, attempts

    head, status = validate_eval_index_candidate("metadata_head", meta_df.iloc[:len(labels)].copy(), labels)
    attempts.append(status)
    if head is not None:
        return head, meta_path, attempts

    if dataset_key == "asvspoof21":
        audio_ids = list_asvspoof2021_audio_ids(config)
        if audio_ids:
            filtered = meta_df[meta_df["utt_id"].astype(str).isin(audio_ids)].copy()
            candidate, status = validate_eval_index_candidate("metadata_filtered_by_existing_flac", filtered, labels)
            attempts.append(status)
            if candidate is not None:
                return candidate, meta_path, attempts
            # The eval dataset class skips rows whose audio file exists but
            # cannot be decoded. If the existing-file candidate is only a few
            # rows longer than results.pkl, this pass reproduces that skip path.
            loadable = filter_asvspoof2021_loadable_audio(meta_df, config)
            candidate, status = validate_eval_index_candidate("metadata_filtered_by_loadable_flac", loadable, labels)
            attempts.append(status)
            if candidate is not None:
                return candidate, meta_path, attempts
        else:
            attempts.append("metadata_filtered_by_existing_flac: no ASVspoof 2021 FLAC folders attached")

    if dataset_key == "asvspoof5":
        # ASVspoof 5 eval results should now carry __metadata__.utt_ids from the
        # evaluation notebook. This fallback is kept for older pickles: first try
        # attached extracted FLAC files, then reproduce the HF tar member order.
        audio_ids = list_asvspoof5_audio_ids_from_files(config)
        if audio_ids:
            filtered = meta_df[meta_df["utt_id"].astype(str).isin(audio_ids)].copy()
            candidate, status = validate_eval_index_candidate("metadata_filtered_by_attached_flac", filtered, labels)
            attempts.append(status)
            if candidate is not None:
                return candidate, meta_path, attempts
        else:
            attempts.append("metadata_filtered_by_attached_flac: no extracted ASVspoof 5 FLAC files attached")

        tar_order = build_asvspoof5_tar_order_index(meta_df, config)
        candidate, status = validate_eval_index_candidate("hf_tar_member_order", tar_order, labels)
        attempts.append(status)
        if candidate is not None:
            return candidate, meta_path, attempts

    detail = "; ".join(attempts)
    raise RuntimeError(f"{dataset_key}: could not reconstruct evaluated utt_ids. Attempts: {detail}")


def ensure_results_have_utt_ids(dataset_key, results, config, result_path):
    """Patch results.pkl with dataset-level utt_ids when they are missing.

    The patched file is written to /kaggle/working/results_with_utt_ids so the
    original read-only Kaggle input remains untouched. Later runs can upload the
    patched file back into the sdd-survey Kaggle dataset.
    """
    labels = labels_from_results(results)
    existing = get_result_utt_ids(results)
    if existing is not None:
        if len(existing) != len(labels):
            raise ValueError(f"{dataset_key}: stored utt_ids length {len(existing):,} != labels length {len(labels):,}")
        return results, result_path, "utt_ids already present"

    if not BACKFILL_UTT_IDS:
        return results, result_path, "utt_ids missing; BACKFILL_UTT_IDS=False"

    eval_index, meta_path, attempts = build_eval_index_from_metadata(dataset_key, config, labels)
    patched = dict(results)
    patched[RESULT_METADATA_KEY] = {
        "dataset": dataset_key,
        "utt_ids": eval_index["utt_id"].astype(str).to_numpy(),
        "label_convention": "1=bonafide,0=spoof",
        "score_convention": "bonafide_probability",
        "source": "backfilled_from_eval_index",
        "metadata_path": str(meta_path),
        "backfill_attempts": attempts,
    }
    patched_path = PATCHED_RESULTS_ROOT / dataset_key / "results.pkl"
    save_pickle_compat(patched, patched_path)

    report_path = PATCHED_RESULTS_ROOT / "alignment_report.csv"
    row = pd.DataFrame([{
        "dataset": dataset_key,
        "source_results": str(result_path),
        "patched_results": str(patched_path),
        "n": int(len(labels)),
        "metadata_path": str(meta_path),
        "status": "patched utt_ids",
        "attempts": " | ".join(attempts),
    }])
    if report_path.exists():
        previous = pd.read_csv(report_path)
        previous = previous[previous["dataset"] != dataset_key]
        row = pd.concat([previous, row], ignore_index=True)
    row.to_csv(report_path, index=False)
    return patched, patched_path, f"utt_ids backfilled -> {patched_path}"


METADATA_PARSERS = {
    "asvspoof19": parse_asvspoof19_metadata,
    "asvspoof21": parse_asvspoof21_metadata,
    "asvspoof5": parse_asvspoof5_metadata,
    "in_the_wild": parse_in_the_wild_metadata,
}


def align_metadata(meta_df, labels):
    labels = np.asarray(labels, dtype=np.int64)
    if meta_df is None or meta_df.empty:
        return None, "metadata missing"
    if len(meta_df) == len(labels):
        aligned = meta_df.reset_index(drop=True).copy()
    elif len(meta_df) > len(labels):
        candidate = meta_df.iloc[:len(labels)].reset_index(drop=True).copy()
        if "metadata_label" in candidate and np.array_equal(candidate["metadata_label"].to_numpy(dtype=np.int64), labels):
            aligned = candidate
        else:
            return None, f"metadata length {len(meta_df):,} does not align with result length {len(labels):,}"
    else:
        return None, f"metadata length {len(meta_df):,} is shorter than result length {len(labels):,}"

    if "metadata_label" in aligned:
        mismatch = int(np.sum(aligned["metadata_label"].to_numpy(dtype=np.int64) != labels))
        if mismatch:
            return None, f"metadata label mismatch in {mismatch:,}/{len(labels):,} rows"
    return aligned.drop(columns=["metadata_label"], errors="ignore"), "metadata aligned"


def join_metadata_by_utt_id(base_df, meta_df):
    """Attach metadata by utterance ID and validate labels for matched rows."""
    if meta_df is None or meta_df.empty or "utt_id" not in meta_df.columns:
        return base_df, "metadata missing or has no utt_id column"
    meta = meta_df.drop_duplicates("utt_id").copy()
    joined = base_df.merge(meta, on="utt_id", how="left", suffixes=("", "_meta"))
    matched = int(joined[[c for c in meta.columns if c != "utt_id"]].notna().any(axis=1).sum())
    status = f"metadata joined by utt_id ({matched:,}/{len(base_df):,} rows matched)"
    if "metadata_label" in joined.columns:
        matched_label = joined["metadata_label"].notna()
        mismatch = int(np.sum(joined.loc[matched_label, "metadata_label"].to_numpy(dtype=np.int64) != joined.loc[matched_label, "label"].to_numpy(dtype=np.int64)))
        if mismatch:
            status += f"; warning: {mismatch:,} matched rows have label mismatch"
        joined = joined.drop(columns=["metadata_label"], errors="ignore")
    return joined, status


def build_dataset_frame(dataset_key, results, config):
    labels = labels_from_results(results)
    n = len(labels)
    utt_ids = get_result_utt_ids(results)
    if utt_ids is None:
        utt_ids = np.asarray([f"{dataset_key}_{i:07d}" for i in range(n)], dtype=str)
        utt_id_status = "utt_ids missing; using synthetic row ids"
    else:
        utt_ids = np.asarray(utt_ids, dtype=str)
        if len(utt_ids) != n:
            raise ValueError(f"{dataset_key}: utt_ids length {len(utt_ids):,} != labels length {n:,}")
        utt_id_status = "utt_ids present"
    df = pd.DataFrame({"row_id": np.arange(n), "utt_id": utt_ids, "label": labels})
    meta_path = first_existing(config.get("metadata_paths", []))
    metadata_status = "metadata path not found"
    if meta_path is not None:
        try:
            meta_df = METADATA_PARSERS[dataset_key](meta_path)
            if get_result_utt_ids(results) is not None:
                df, metadata_status = join_metadata_by_utt_id(df, meta_df)
            else:
                aligned, metadata_status = align_metadata(meta_df, labels)
                if aligned is not None:
                    base_cols = [c for c in aligned.columns if c != "label"]
                    df = pd.concat([df.drop(columns=["utt_id"]), aligned[base_cols].reset_index(drop=True)], axis=1)
                    if "utt_id" not in df.columns:
                        df["utt_id"] = utt_ids
        except Exception as exc:
            metadata_status = f"metadata parse failed: {exc}"
    metadata_status = f"{utt_id_status}; {metadata_status}"

    models = model_names_from_results(results, n)
    for model in models:
        scores = np.asarray(results[model]["scores"], dtype=np.float64)
        eer, threshold = compute_eer_threshold(scores, labels)
        df[f"{model}_score"] = scores
        df[f"{model}_pred"] = (scores >= threshold).astype(np.int64)
        df[f"{model}_correct"] = df[f"{model}_pred"].to_numpy(dtype=np.int64) == labels
    return df, models, metadata_status, meta_path

## Run per-dataset analysis

The loop below analyzes every dataset with an available `results.pkl`. Missing models are skipped automatically. Metadata-dependent breakdowns join by `utt_id` when available; if `utt_id` is missing, the notebook first tries to backfill it from the same protocol/audio-index logic used by evaluation.

In [ ]:
def metrics_for_dataset(df, models):
    labels = df["label"].to_numpy(dtype=np.int64)
    metrics = {}
    rows = []
    for model in models:
        scores = df[f"{model}_score"].to_numpy(dtype=np.float64)
        eer, threshold = compute_eer_threshold(scores, labels)
        item = compute_operating_metrics(scores, labels, threshold)
        ece, _ = expected_calibration_error(scores, labels)
        item["eer"] = float(eer)
        item["ece"] = float(ece)
        metrics[model] = item
        rows.append({"model": model, **item})
    return metrics, pd.DataFrame(rows)


def grouped_eer_for_dataset(df, models, fields, metrics):
    grouped = {}
    for field in fields:
        if field not in df.columns:
            continue
        rows = []
        for group_value, group_df in df.groupby(field, dropna=False):
            labels = group_df["label"].to_numpy(dtype=np.int64)
            mode = "within_group"
            eval_df = group_df
            # Attack tags are often spoof-only. To make those groups usable for
            # EER, compare that spoof subset against all bonafide utterances in
            # the dataset. This is the standard "attack-specific EER" view used
            # for ASVspoof-style error analysis.
            if len(np.unique(labels)) < 2 and np.all(labels == 0):
                bona_df = df[df["label"] == 1]
                if bona_df.empty:
                    continue
                eval_df = pd.concat([bona_df, group_df], ignore_index=True)
                labels = eval_df["label"].to_numpy(dtype=np.int64)
                mode = "spoof_group_vs_all_bonafide"
            elif len(np.unique(labels)) < 2:
                continue
            for model in models:
                scores = eval_df[f"{model}_score"].to_numpy(dtype=np.float64)
                eer, _ = compute_eer_threshold(scores, labels)
                threshold = metrics[model]["threshold"]
                op = compute_operating_metrics(scores, labels, threshold)
                rows.append({
                    "field": field,
                    "group": str(group_value),
                    "model": model,
                    "mode": mode,
                    "n": int(len(eval_df)),
                    "n_group": int(len(group_df)),
                    "n_bonafide": int(np.sum(labels == 1)),
                    "n_spoof": int(np.sum(labels == 0)),
                    "eer": float(eer),
                    "far": op["far"],
                    "frr": op["frr"],
                    "fp": op["fp"],
                    "fn": op["fn"],
                })
        if rows:
            grouped[field] = pd.DataFrame(rows).sort_values(["eer", "n"], ascending=[False, False]).reset_index(drop=True)
    return grouped


def score_correlation_for_dataset(df, models):
    score_cols = [f"{model}_score" for model in models]
    corr = df[score_cols].corr(method="spearman")
    corr.index = models
    corr.columns = models
    return corr


def failure_overlap_for_dataset(df, models):
    rows = []
    error_sets = {
        model: set(df.index[~df[f"{model}_correct"].astype(bool)].tolist())
        for model in models
    }
    for left in models:
        row = {"model": left}
        for right in models:
            union = error_sets[left] | error_sets[right]
            inter = error_sets[left] & error_sets[right]
            row[right] = float(len(inter) / len(union)) if union else 0.0
        rows.append(row)
    return pd.DataFrame(rows).set_index("model")


def hard_errors_for_dataset(df, models, metrics, top_k=25):
    hard = {}
    csv_rows = []
    error_count = np.zeros(len(df), dtype=np.int64)
    for model in models:
        pred = df[f"{model}_pred"].to_numpy(dtype=np.int64)
        labels = df["label"].to_numpy(dtype=np.int64)
        scores = df[f"{model}_score"].to_numpy(dtype=np.float64)
        error_count += (pred != labels).astype(np.int64)

        fp = df[(labels == 0) & (pred == 1)].copy()
        fn = df[(labels == 1) & (pred == 0)].copy()
        fp = fp.assign(error_type="false_positive", model=model, confidence=fp[f"{model}_score"])
        fn = fn.assign(error_type="false_negative", model=model, confidence=1.0 - fn[f"{model}_score"])
        fp = fp.sort_values([f"{model}_score", "utt_id"], ascending=[False, True]).head(top_k)
        fn = fn.sort_values([f"{model}_score", "utt_id"], ascending=[True, True]).head(top_k)

        cols = ["utt_id", "label"]
        metadata_cols = [c for c in ("speaker", "attack", "attack_tag", "codec", "vocoder", "condition", "duration", "duration_bucket") if c in df.columns]
        export_cols = cols + metadata_cols + [f"{model}_score", "error_type", "confidence"]
        top_fp = fp[export_cols].to_dict("records") if not fp.empty else []
        top_fn = fn[export_cols].to_dict("records") if not fn.empty else []
        hard[model] = {"top_fp": top_fp, "top_fn": top_fn}
        csv_rows.extend(top_fp)
        csv_rows.extend(top_fn)

    universal_df = df.copy()
    universal_df["failed_model_count"] = error_count
    universal_df = universal_df[universal_df["failed_model_count"] >= max(2, min(len(models), math.ceil(len(models) * 0.75)))]
    universal_df = universal_df.sort_values(["failed_model_count", "utt_id"], ascending=[False, True]).head(top_k)
    hard["_universal"] = universal_df[["utt_id", "label", "failed_model_count"]].to_dict("records")
    for row in hard["_universal"]:
        row["model"] = "_universal"
        row["error_type"] = "shared_error"
        csv_rows.append(row)
    return hard, pd.DataFrame(csv_rows)


def plot_score_distributions(df, models, out_dir):
    if plt is None:
        return
    out_dir.mkdir(parents=True, exist_ok=True)
    if not models:
        return
    ncols = 2
    nrows = math.ceil(len(models) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3.2 * nrows), squeeze=False)
    labels = df["label"].to_numpy(dtype=np.int64)
    for ax, model in zip(axes.ravel(), models):
        scores = df[f"{model}_score"].to_numpy(dtype=np.float64)
        ax.hist(scores[labels == 0], bins=60, alpha=0.55, density=True, label="spoof", color="#d95f02")
        ax.hist(scores[labels == 1], bins=60, alpha=0.55, density=True, label="bonafide", color="#1b9e77")
        ax.set_title(model)
        ax.set_xlabel("bonafide score")
        ax.set_ylabel("density")
        ax.legend()
    for ax in axes.ravel()[len(models):]:
        ax.axis("off")
    fig.tight_layout()
    fig.savefig(out_dir / "score_distributions.png", dpi=160)
    plt.close(fig)


def plot_reliability(df, models, out_dir):
    if plt is None:
        return
    out_dir.mkdir(parents=True, exist_ok=True)
    if not models:
        return
    fig, ax = plt.subplots(figsize=(6, 5))
    labels = df["label"].to_numpy(dtype=np.int64)
    for model in models:
        _, rel = expected_calibration_error(df[f"{model}_score"].to_numpy(dtype=np.float64), labels)
        valid = rel[rel["n"] > 0]
        ax.plot(valid["mean_score"], valid["frac_bonafide"], marker="o", linewidth=1.5, label=model)
    ax.plot([0, 1], [0, 1], linestyle="--", color="black", linewidth=1)
    ax.set_xlabel("mean predicted bonafide score")
    ax.set_ylabel("empirical bonafide fraction")
    ax.set_title("Reliability diagram")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(out_dir / "reliability.png", dpi=160)
    plt.close(fig)


def plot_grouped_eer(grouped, out_dir):
    if plt is None:
        return
    out_dir.mkdir(parents=True, exist_ok=True)
    for field, table in grouped.items():
        top_groups = table.groupby("group")["eer"].max().sort_values(ascending=False).head(20).index.tolist()
        plot_df = table[table["group"].isin(top_groups)].copy()
        if plot_df.empty:
            continue
        pivot = plot_df.pivot_table(index="group", columns="model", values="eer", aggfunc="mean")
        ax = pivot.plot(kind="bar", figsize=(max(10, 0.55 * len(pivot)), 5))
        ax.set_ylabel("EER (%)")
        ax.set_title(f"EER by {field}")
        ax.legend(fontsize=8)
        fig = ax.get_figure()
        fig.tight_layout()
        fig.savefig(out_dir / f"grouped_eer_{field}.png", dpi=160)
        plt.close(fig)


def plot_correlation(corr, out_dir):
    if plt is None:
        return
    out_dir.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(corr.to_numpy(dtype=float), vmin=-1, vmax=1, cmap="coolwarm")
    ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(corr.index)), corr.index)
    for i in range(len(corr.index)):
        for j in range(len(corr.columns)):
            ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
    ax.set_title("Spearman score correlation")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(out_dir / "score_correlation.png", dpi=160)
    plt.close(fig)


def export_dataset_artifacts(dataset_key, analysis, models, hard_errors_table):
    out_dir = OUTPUT_ROOT / dataset_key
    plots_dir = out_dir / "plots"
    out_dir.mkdir(parents=True, exist_ok=True)
    plots_dir.mkdir(parents=True, exist_ok=True)

    with open(out_dir / "error_analysis.pkl", "wb") as handle:
        pickle.dump(analysis, handle, protocol=pickle.HIGHEST_PROTOCOL)
    pd.DataFrame.from_dict(analysis["metrics"], orient="index").rename_axis("model").reset_index().to_csv(out_dir / "metrics.csv", index=False)
    analysis["score_correlation"].to_csv(out_dir / "score_correlation.csv")
    analysis["failure_overlap"].to_csv(out_dir / "failure_overlap.csv")
    if hard_errors_table is not None and not hard_errors_table.empty:
        hard_errors_table.to_csv(out_dir / "hard_errors.csv", index=False)
    for field, table in analysis["grouped_eer"].items():
        table.to_csv(out_dir / f"grouped_eer_{field}.csv", index=False)

    plot_score_distributions(analysis["df"], models, plots_dir)
    plot_reliability(analysis["df"], models, plots_dir)
    plot_grouped_eer(analysis["grouped_eer"], plots_dir)
    plot_correlation(analysis["score_correlation"], plots_dir)
    print(f"exported {dataset_key}: {out_dir}")


def analyze_dataset(dataset_key, config):
    result_path = None
    results = None
    skipped_paths = []
    for candidate in config["result_paths"]:
        candidate = Path(candidate)
        if not candidate.exists():
            continue
        candidate_results = load_pickle_compat(candidate)
        if result_metadata_matches_config(dataset_key, candidate_results, config):
            result_path = candidate
            results = candidate_results
            break
        skipped_paths.append(str(candidate))
    if result_path is None or results is None:
        print(f"{dataset_key}: results.pkl not found; skipping")
        if skipped_paths:
            print(f"{dataset_key}: skipped incompatible results files: {skipped_paths}")
        DATASET_STATUS[dataset_key] = {"status": "skipped; results.pkl not found"}
        return None
    print(f"\n## {config['display']}")
    print(f"loading results: {result_path}")
    if skipped_paths:
        print(f"skipped incompatible results files: {skipped_paths}")
    try:
        results, result_path, utt_id_status = ensure_results_have_utt_ids(dataset_key, results, config, result_path)
        print(f"utt_id status: {utt_id_status}")
    except Exception as exc:
        print(f"utt_id backfill warning: {exc}")
        utt_id_status = f"utt_id backfill failed: {exc}"
    df, models, metadata_status, meta_path = build_dataset_frame(dataset_key, results, config)
    print(f"models: {models}")
    print(f"metadata: {metadata_status}; path={meta_path}")
    DATASET_STATUS[dataset_key] = {
        "status": metadata_status,
        "utt_id_status": utt_id_status,
        "result_path": str(result_path),
        "metadata_path": str(meta_path) if meta_path is not None else None,
        "n_rows": int(len(df)),
        "n_models": int(len(models)),
    }
    metrics, metrics_df = metrics_for_dataset(df, models)
    grouped = grouped_eer_for_dataset(df, models, config.get("metadata_fields", []), metrics)
    corr = score_correlation_for_dataset(df, models)
    overlap = failure_overlap_for_dataset(df, models)
    hard, hard_table = hard_errors_for_dataset(df, models, metrics, TOP_K_HARD_ERRORS)

    analysis = {
        "dataset": dataset_key,
        "df": df,
        "metrics": metrics,
        "grouped_eer": grouped,
        "score_correlation": corr,
        "failure_overlap": overlap,
        "hard_errors": hard,
    }
    export_dataset_artifacts(dataset_key, analysis, models, hard_table)
    display(metrics_df.sort_values("eer"))
    return analysis


ANALYSES = {}
for dataset_key, config in DATASETS.items():
    ANALYSES[dataset_key] = analyze_dataset(dataset_key, config)

## Cross-dataset synthesis

This cell writes cross-dataset EER tables, generalization gaps against ASVspoof 2019 when available, robustness summaries, and a deterministic `report.md`.

In [ ]:
def build_synthesis(analyses):
    rows = []
    for dataset_key, analysis in analyses.items():
        if analysis is None:
            continue
        for model, item in analysis["metrics"].items():
            rows.append({
                "dataset": dataset_key,
                "display": DATASETS[dataset_key]["display"],
                "model": model,
                "eer": item["eer"],
                "threshold": item["threshold"],
                "far": item["far"],
                "frr": item["frr"],
                "accuracy": item["accuracy"],
                "ece": item["ece"],
                "n": item["n"],
            })
    metrics_all = pd.DataFrame(rows)
    if metrics_all.empty:
        return metrics_all, pd.DataFrame(), []

    eer_table = metrics_all.pivot_table(index="model", columns="dataset", values="eer", aggfunc="first")
    if "asvspoof19" in eer_table.columns:
        gap = eer_table.subtract(eer_table["asvspoof19"], axis=0)
    else:
        gap = pd.DataFrame(index=eer_table.index)
    robustness = pd.DataFrame({
        "mean_eer": eer_table.mean(axis=1),
        "std_eer": eer_table.std(axis=1),
        "datasets_evaluated": eer_table.notna().sum(axis=1),
    }).sort_values(["mean_eer", "std_eer"])

    out_dir = OUTPUT_ROOT / "synthesis"
    plots_dir = out_dir / "plots"
    out_dir.mkdir(parents=True, exist_ok=True)
    plots_dir.mkdir(parents=True, exist_ok=True)
    metrics_all.to_csv(out_dir / "all_metrics.csv", index=False)
    eer_table.to_csv(out_dir / "eer_table.csv")
    gap.to_csv(out_dir / "generalization_gap.csv")
    robustness.to_csv(out_dir / "robustness_summary.csv")

    if plt is not None:
        ax = eer_table.plot(kind="bar", figsize=(11, 5))
        ax.set_ylabel("EER (%)")
        ax.set_title("EER across datasets")
        ax.legend(title="dataset", fontsize=8)
        fig = ax.get_figure()
        fig.tight_layout()
        fig.savefig(plots_dir / "eer_table.png", dpi=160)
        plt.close(fig)

    def table_to_markdown(table):
        try:
            return table.to_markdown()
        except Exception:
            return "```\n" + table.to_string() + "\n```"

    report_lines = [
        "# Error Analysis Report",
        "",
        "This report is generated from the current `results.pkl` files and available protocol metadata.",
        "Labels use `1=bonafide`, `0=spoof`; model scores are bonafide probabilities.",
        "",
        "t-DCF is not reported because this repository currently stores CM scores only, not ASV-side scores.",
        "To add t-DCF later, attach the official ASVspoof ASV scores and use the challenge evaluation script.",
        "",
        "## Dataset Status",
        "",
    ]
    for dataset_key, analysis in analyses.items():
        if analysis is None:
            report_lines.append(f"- `{dataset_key}`: skipped; results.pkl not found")
            continue
        status = DATASET_STATUS.get(dataset_key, {})
        report_lines.append(
            f"- `{dataset_key}`: {status.get('n_rows', len(analysis['df'])):,} rows, "
            f"{status.get('n_models', len(analysis['metrics']))} models, "
            f"{status.get('utt_id_status', 'utt_id status unknown')}; "
            f"{status.get('status', 'metadata status unknown')}"
        )

    report_lines.extend(["", "## EER Summary", "", table_to_markdown(eer_table.round(4)), ""])
    report_lines.extend(["## Generalization Gap vs ASVspoof 2019", "", table_to_markdown(gap.round(4)), ""])
    report_lines.extend(["## Robustness Summary", "", table_to_markdown(robustness.round(4)), ""])
    report_lines.extend(["## Notes", "", "- Add human interpretation here after reviewing grouped errors and hard examples.", ""])
    (OUTPUT_ROOT / "report.md").write_text("\n".join(report_lines), encoding="utf-8")
    print(f"wrote synthesis: {out_dir}")
    return eer_table, gap, report_lines


EER_TABLE, GENERALIZATION_GAP, REPORT_LINES = build_synthesis(ANALYSES)
display(EER_TABLE)
display(GENERALIZATION_GAP)

## Optional spectrogram scaffold

Spectrograms require the large audio datasets, so they are disabled by default. Use `hard_errors.csv` to select utterances before enabling this section.

In [ ]:
# Optional qualitative cell. It is intentionally off by default because audio datasets are large.
# Set RUN_SPECTROGRAMS=True and attach the relevant audio dataset before extending this cell.
if RUN_SPECTROGRAMS:
    print("Spectrogram export is not wired by default. Use hard_errors.csv utt_ids to locate audio and inspect selected cases.")
else:
    print("RUN_SPECTROGRAMS=False; skipping optional spectrogram export")

## Optional zip export

The zip step is last so partial dataset artifacts remain available even if compression fails.

In [ ]:
if CREATE_ZIP:
    zip_path = OUTPUT_ROOT.parent / "error_analysis_artifacts.zip"
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for root in (OUTPUT_ROOT, PATCHED_RESULTS_ROOT):
            if not root.exists():
                continue
            for path in root.rglob("*"):
                if path.is_file():
                    zf.write(path, path.relative_to(OUTPUT_ROOT.parent))
    print(f"created {zip_path}")
else:
    print("CREATE_ZIP=False; skipping zip export")